## Bronze Table: Locations

Ingests data from the `locations` volume containing **districts**, **municipalities**, **parishes** and **localities** and overwrites it to the respective targets in `bronze` Delta table with the date and timestamp of latest ingestion.

Columns containing only null values are dropped to prevent schema conflicts.

In [0]:
import pandas as pd
from datetime import datetime

In [0]:
# Define the four location datasets
locations = [
    ("districts", "districts.json"),
    ("municipalities", "municipalities.json"),
    ("parishes", "parishes.json"),
    ("localities", "localities.json"),
]

volume_path = "/Volumes/trips_carris_metropolitana/bronze/locations"

for name, filename in locations:
    file_path = f"{volume_path}/{filename}"
    
    # Read JSON
    df = pd.read_json(file_path)
    
    # Drop fully null columns
    df = df.dropna(axis=1, how='all')
    
    # Convert empty lists to empty string (avoids Spark NullType issues)
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list) and len(x) == 0).all():
            df[col] = ""
    
    # Add ingestion metadata
    df["ingestion_date"] = datetime.now().strftime("%Y-%m-%d")
    df["ingestion_timestamp"] = datetime.now()
    
    # Write to bronze table (overwrite)
    spark_df = spark.createDataFrame(df)
    table_name = f"trips_carris_metropolitana.bronze.{name}"
    spark_df.write.mode("overwrite").saveAsTable(table_name)
    
    print(f"Overwrote {len(df)} records into {table_name}")

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.districts LIMIT 5;

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.municipalities LIMIT 5;

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.parishes LIMIT 5;

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.localities LIMIT 5;